# 📓 Notebook 04: Train & Đánh giá 2 thuật toán tự code

**Mục tiêu:**
1. Train Decision Tree với các tham số khác nhau
2. Train Naive Bayes và đánh giá
3. Confusion Matrix cho từng model
4. Dự đoán mẫu cụ thể (ví dụ bệnh nhân)
5. K-Fold Cross Validation

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import (
    train_test_split, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, k_fold_cross_validation
)
from src.models.decision_tree import DecisionTree
from src.models.naive_bayes import NaiveBayes

print("✅ Import thành công")

In [ ]:
# ===== LOAD DATA =====
df = pd.read_csv('../data/heart_preprocessed.csv')
X = df.drop('HeartDisease', axis=1).values
y = df['HeartDisease'].values

print(f"📊 Dữ liệu: {len(df)} samples, {X.shape[1]} features")
print(f"   Target: 0={sum(y==0)}, 1={sum(y==1)}")

# Chia train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"   Train: {len(X_train)}, Test: {len(X_test)}")

---
## 📌 Phần A: Decision Tree - Tuning tham số

In [ ]:
# ===== DECISION TREE: TUNING THAM SỐ =====
print("=" * 60)
print("  DECISION TREE - TUNING THAM SỐ")
print("=" * 60)

# Thử các tổ hợp tham số
param_grid = [
    {'max_depth': 3, 'min_samples_split': 2},
    {'max_depth': 5, 'min_samples_split': 2},
    {'max_depth': 5, 'min_samples_split': 10},
    {'max_depth': 10, 'min_samples_split': 2},
    {'max_depth': 10, 'min_samples_split': 20},
    {'max_depth': 15, 'min_samples_split': 20},
    {'max_depth': 20, 'min_samples_split': 50},
]

print(f"\n{'Params':<35} {'Accuracy':<10} {'Precision':<10} {'Recall':<10} {'F1':<10}")
print("=" * 75)

best_dt = None
best_f1_dt = 0
best_params_dt = None

for params in param_grid:
    dt = DecisionTree(**params)
    dt.fit(X_train, y_train)
    y_pred = dt.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    label = f"max_depth={params['max_depth']}, min_samples={params['min_samples_split']}"
    print(f"{label:<35} {acc:.4f}     {prec:.4f}     {rec:.4f}     {f1:.4f}")
    
    if f1 > best_f1_dt:
        best_f1_dt = f1
        best_dt = dt
        best_params_dt = params

print(f"\n✅ Best params: {best_params_dt} (F1={best_f1_dt:.4f})")

In [ ]:
# ===== DECISION TREE: CONFUSION MATRIX =====
y_pred_dt = best_dt.predict(X_test)
cm_dt = confusion_matrix(y_test, y_pred_dt)

print("📊 DECISION TREE - Confusion Matrix:")
print(f"{'':>12} {'Predicted':>20}")
print(f"{'':>12} {'0':>8} {'1':>8}")
print(f"{'Actual 0':>12} {cm_dt[0,0]:>8} {cm_dt[0,1]:>8}")
print(f"{'Actual 1':>12} {cm_dt[1,0]:>8} {cm_dt[1,1]:>8}")

# Vẽ heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Không bệnh', 'Có bệnh'],
            yticklabels=['Không bệnh', 'Có bệnh'])
plt.title(f'Decision Tree (max_depth={best_params_dt["max_depth"]})')
plt.ylabel('Thực tế')
plt.xlabel('Dự đoán')
plt.show()

In [ ]:
# ===== DECISION TREE: K-FOLD CV =====
print("📊 DECISION TREE - K-Fold Cross Validation (k=5):")
print("-" * 40)

accuracies = k_fold_cross_validation(
    X, y, DecisionTree,
    k=5, random_state=42,
    **best_params_dt
)

for i, acc in enumerate(accuracies):
    print(f"   Fold {i+1}: {acc:.4f}")

print(f"\n   Mean Accuracy: {np.mean(accuracies):.4f}")
print(f"   Std:           {np.std(accuracies):.4f}")

---
## 📌 Phần B: Naive Bayes - Đánh giá

In [ ]:
# ===== NAIVE BAYES: TRAIN & EVALUATE =====
print("=" * 60)
print("  NAIVE BAYES - ĐÁNH GIÁ")
print("=" * 60)

nb = NaiveBayes()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

acc_nb = accuracy_score(y_test, y_pred_nb)
prec_nb = precision_score(y_test, y_pred_nb)
rec_nb = recall_score(y_test, y_pred_nb)
f1_nb = f1_score(y_test, y_pred_nb)

print(f"\n{'Metric':<15} {'Value':<10}")
print("-" * 25)
print(f"{'Accuracy':<15} {acc_nb:.4f}")
print(f"{'Precision':<15} {prec_nb:.4f}")
print(f"{'Recall':<15} {rec_nb:.4f}")
print(f"{'F1-Score':<15} {f1_nb:.4f}")

In [ ]:
# ===== NAIVE BAYES: CONFUSION MATRIX =====
cm_nb = confusion_matrix(y_test, y_pred_nb)

print("📊 NAIVE BAYES - Confusion Matrix:")
print(f"{'':>12} {'Predicted':>20}")
print(f"{'':>12} {'0':>8} {'1':>8}")
print(f"{'Actual 0':>12} {cm_nb[0,0]:>8} {cm_nb[0,1]:>8}")
print(f"{'Actual 1':>12} {cm_nb[1,0]:>8} {cm_nb[1,1]:>8}")

# Vẽ heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Không bệnh', 'Có bệnh'],
            yticklabels=['Không bệnh', 'Có bệnh'])
plt.title('Naive Bayes')
plt.ylabel('Thực tế')
plt.xlabel('Dự đoán')
plt.show()

In [ ]:
# ===== NAIVE BAYES: K-FOLD CV =====
print("📊 NAIVE BAYES - K-Fold Cross Validation (k=5):")
print("-" * 40)

accuracies_nb = k_fold_cross_validation(
    X, y, NaiveBayes,
    k=5, random_state=42
)

for i, acc in enumerate(accuracies_nb):
    print(f"   Fold {i+1}: {acc:.4f}")

print(f"\n   Mean Accuracy: {np.mean(accuracies_nb):.4f}")
print(f"   Std:           {np.std(accuracies_nb):.4f}")

---
## 📌 Phần C: So sánh 2 models

In [ ]:
# ===== SO SÁNH 2 MODELS =====
print("=" * 60)
print("  SO SÁNH DECISION TREE vs NAIVE BAYES")
print("=" * 60)

print(f"\n{'Model':<25} {'Accuracy':<10} {'Precision':<10} {'Recall':<10} {'F1':<10}")
print("=" * 65)

dt_acc = accuracy_score(y_test, y_pred_dt)
dt_prec = precision_score(y_test, y_pred_dt)
dt_rec = recall_score(y_test, y_pred_dt)
dt_f1 = f1_score(y_test, y_pred_dt)

print(f"{'Decision Tree':<25} {dt_acc:.4f}     {dt_prec:.4f}     {dt_rec:.4f}     {dt_f1:.4f}")
print(f"{'Naive Bayes':<25} {acc_nb:.4f}     {prec_nb:.4f}     {rec_nb:.4f}     {f1_nb:.4f}")

# Biểu đồ so sánh
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
dt_scores = [dt_acc, dt_prec, dt_rec, dt_f1]
nb_scores = [acc_nb, prec_nb, rec_nb, f1_nb]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, dt_scores, width, label='Decision Tree', color='steelblue')
bars2 = ax.bar(x + width/2, nb_scores, width, label='Naive Bayes', color='seagreen')

ax.set_xlabel('Metrics')
ax.set_ylabel('Score')
ax.set_title('So sánh Decision Tree vs Naive Bayes')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, 1)

# Thêm số trên cột
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')

plt.tight_layout()
plt.show()

---
## 📌 Phần D: Dự đoán mẫu cụ thể

In [ ]:
# ===== DỰ ĐOÁN MẪU CỤ THỂ =====
print("=" * 70)
print("  VÍ DỤ DỰ ĐOÁN TRÊN BỆNH NHÂN CỤ THỂ")
print("=" * 70)

# Load data gốc để lấy thông tin
df_raw = pd.read_csv('../data/heart.csv')

# Lấy index test set
np.random.seed(42)
n = len(df_raw)
indices = np.random.permutation(n)
test_size = int(n * 0.2)
test_idx = indices[:test_size]

# Chọn 3 mẫu để dự đoán
sample_indices = [0, 10, 20]

for idx in sample_indices:
    patient = df_raw.iloc[test_idx[idx]]
    print()
    print(f"--- Bệnh nhân {idx+1} ---")
    print(f"  Tuổi: {patient['Age']}, Giới: {patient['Sex']}")
    print(f"  Đau ngực: {patient['ChestPainType']}, Huyết áp: {patient['RestingBP']}")
    print(f"  Cholesterol: {patient['Cholesterol']}, MaxHR: {patient['MaxHR']}")
    print(f"  Đau thắt khi tập: {patient['ExerciseAngina']}, Oldpeak: {patient['Oldpeak']}")
    print(f"  ST_Slope: {patient['ST_Slope']}")
    print(f"  Thực tế: {'Có bệnh' if y_test[idx] == 1 else 'Không bệnh'}")
    
    pred_dt = best_dt.predict(X_test[idx].reshape(1, -1))[0]
    pred_nb = nb.predict(X_test[idx].reshape(1, -1))[0]
    
    dt_correct = '✅' if pred_dt == y_test[idx] else '❌'
    nb_correct = '✅' if pred_nb == y_test[idx] else '❌'
    
    print(f"  Decision Tree: {'Có bệnh' if pred_dt == 1 else 'Không bệnh'} {dt_correct}")
    print(f"  Naive Bayes:   {'Có bệnh' if pred_nb == 1 else 'Không bệnh'} {nb_correct}")

In [ ]:
# ===== TỔNG KẾT =====
print("=" * 60)
print("  TỔNG KẾT KẾT QUẢ")
print("=" * 60)

print(f"\n{'Model':<25} {'Accuracy':<10} {'Precision':<10} {'Recall':<10} {'F1':<10} {'K-Fold Mean':<12}")
print("=" * 77)
print(f"{'Decision Tree':<25} {dt_acc:.4f}     {dt_prec:.4f}     {dt_rec:.4f}     {dt_f1:.4f}     {np.mean(accuracies):.4f}")
print(f"{'Naive Bayes':<25} {acc_nb:.4f}     {prec_nb:.4f}     {rec_nb:.4f}     {f1_nb:.4f}     {np.mean(accuracies_nb):.4f}")

print(f"\n✅ Hoàn thành đánh giá 2 thuật toán!")